# Занятие 3 — Преобразование и объединение таблиц

## Подготовка

In [ ]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = DATA
decisions = pd.read_csv(DATA / "decisions.csv", dtype={"lei":"string", "county_code":"string"})
organizations = pd.read_csv(LOCAL / "organizations.csv", dtype={"lei":"string"})
print("pandas", pd.__version__, "записей", len(decisions), "организаций", len(organizations))

## B02 — слайд 2: Таблицы и смысл строки

In [ ]:
counts = pd.read_csv(LOCAL / "outcome_counts.csv", dtype={"lei": "string"})
print(counts.shape, counts["n"].sum())
counts.head(6)

## B04 — слайд 4: Счётчики по организации и исходу

In [ ]:
decisions.head()

In [ ]:
keys_long = ["lei", "activity_year", "action_taken"]
long = decisions.groupby(keys_long, dropna=False).size().rename("n").reset_index()
assert not long.duplicated(keys_long).any()
assert long["n"].sum() == len(decisions)
long.head(6)

## B05 — слайд 5: Исходы в отдельных столбцах

In [ ]:
wide_input = pd.read_csv(LOCAL / "outcome_wide.csv", dtype={"lei": "string"})
print(wide_input.shape)
wide_input.head(4)

## B06 — слайд 6: Разворот через pivot

In [ ]:
wide = long.pivot(index=["lei", "activity_year"], columns="action_taken", values="n")
print(wide.shape, wide.index.names, wide.columns.tolist())
wide.head(4)

## B07 — слайд 7: Когда для ячейки есть два значения

In [ ]:
replayed_long = pd.concat([long, long.iloc[[0]]], ignore_index=True)
try:
    replayed_long.pivot(index=["lei", "activity_year"], columns="action_taken", values="n")
except ValueError as error:
    print(type(error).__name__, "неоднозначная ячейка")

In [ ]:
wrong_wide = replayed_long.pivot_table(index=["lei", "activity_year"], columns="action_taken", values="n", aggfunc="sum", fill_value=0)
print("После повторной доставки:", wrong_wide.sum().sum())
assert wrong_wide.sum().sum() > 6830

## B08 — слайд 8: Обратный разворот через melt

In [ ]:
outcome_columns = ["issued", "approved_not_accepted", "denied"]
restored = wide_input.melt(id_vars=["lei", "activity_year"], value_vars=outcome_columns, var_name="outcome", value_name="n")
print(restored.shape, restored["n"].sum())
assert len(restored) == 942
assert restored["n"].gt(0).sum() == 500
restored.head(6)

## B09 — слайд 9: Ноль отсутствующего исхода

In [ ]:
wide_complete = wide.reindex(columns=[1, 2, 3]).fillna(0).astype("int64")
wide_complete = wide_complete.rename(columns={1:"issued", 2:"approved_not_accepted", 3:"denied"}).reset_index()
print(wide_complete.dtypes)
assert wide_complete[outcome_columns].sum().sum() == 6830

In [ ]:
wide_complete.head()

## B10 — слайд 10: Проверка обратного преобразования

In [ ]:
code_map = {"issued":1, "approved_not_accepted":2, "denied":3}
restored["action_taken"] = restored["outcome"].map(code_map)
roundtrip = restored.loc[restored["n"].gt(0), keys_long + ["n"]]
expected = long.sort_values(keys_long).reset_index(drop=True)
actual = roundtrip.sort_values(keys_long).reset_index(drop=True)
assert_frame_equal(actual, expected)
print("Форма изменена; исходные счётчики восстановлены")

## B11 — слайд 11: Ключ справочника

In [ ]:
org_keys = ["lei", "activity_year"]
print(organizations.shape)
assert not organizations.duplicated(org_keys).any()
assert not organizations[org_keys].isna().any().any()
assert decisions["source_row"].is_unique
organizations.head(4)

## B13 — слайд 13: Соединение на двух записях

In [ ]:
example_lei = organizations.iloc[0]["lei"]
left_example = decisions.loc[decisions["lei"].eq(example_lei), ["source_row", "lei", "activity_year", "action_taken"]].head(2)
right_example = organizations.loc[organizations["lei"].eq(example_lei)]
print(left_example.to_string(index=False))
print(right_example.to_string(index=False))

In [ ]:
hand_join = left_example.merge(right_example, on=org_keys, how="left", validate="many_to_one")
hand_join[["source_row", "organization_name", "action_taken"]]

## B14 — слайд 14: left и inner при неполном справочнике

In [ ]:
partial_orgs = organizations.loc[~organizations["lei"].eq(example_lei)].copy()
left_partial = decisions.merge(partial_orgs, on=org_keys, how="left", validate="many_to_one", indicator=True)
inner_partial = decisions.merge(partial_orgs, on=org_keys, how="inner", validate="many_to_one")
print(len(left_partial), len(inner_partial), left_partial["_merge"].value_counts().to_dict())
assert len(left_partial) == 6830 and len(inner_partial) == 6805

## B15 — слайд 15: Полное соединение и проверка строк

In [ ]:
enriched = decisions.merge(organizations, on=org_keys, how="left", validate="many_to_one", indicator=True)
print(enriched.shape)
assert len(enriched) == len(decisions)
assert enriched["source_row"].is_unique
assert set(enriched["source_row"]) == set(decisions["source_row"])
enriched.head(5)

## B16 — слайд 16: Повтор справочника умножает строки

In [ ]:
replayed_orgs = pd.concat([organizations, organizations.iloc[[0]]], ignore_index=True)
wrong_join = decisions.merge(replayed_orgs, on=org_keys, how="left")
print("Строк:", len(wrong_join), "лишних:", wrong_join["source_row"].duplicated().sum())
assert len(wrong_join) == 6855

In [ ]:
try:
    decisions.merge(replayed_orgs, on=org_keys, how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print(type(error).__name__, "справа нарушена уникальность")

## B17 — слайд 17: Каким записям не нашлось имени

In [ ]:
print(enriched["_merge"].value_counts(dropna=False))
assert enriched["_merge"].eq("both").all()
unmatched = left_partial.loc[left_partial["_merge"].eq("left_only"), ["source_row"] + org_keys]
print(unmatched.shape)
assert len(unmatched) == 25
unmatched.head(3)

## B18 — слайд 18: Пропуски в ключе

In [ ]:
unknown_left = left_example.copy()
unknown_right = right_example.copy()
unknown_left["lei"] = pd.NA
unknown_right["lei"] = pd.NA
null_pairs = unknown_left.merge(unknown_right, on=org_keys, how="inner", indicator=True)
print("Техническая иллюстрация совпавших неизвестных ключей:", len(null_pairs))
assert len(null_pairs) == 2

## B19 — слайд 19: Какие поля брать из справочника

In [ ]:
name_lookup = organizations[org_keys + ["organization_name"]].copy()
analysis_table = decisions.merge(name_lookup, on=org_keys, how="left", validate="many_to_one")
print(analysis_table.columns.tolist())
assert "source_count" not in analysis_table

## B20 — слайд 20: Профиль организаций с именами

In [ ]:
prepared = decisions.copy()
prepared["is_approved"] = prepared["action_taken"].isin([1,2])
profile = prepared.groupby(org_keys, dropna=False).agg(n_decisions=("action_taken","size"), n_approved=("is_approved","sum")).reset_index()
profile["approval_rate"] = profile["n_approved"] / profile["n_decisions"]

In [ ]:
named_profile = profile.merge(name_lookup, on=org_keys, how="left", validate="one_to_one")
assert len(named_profile) == 314
assert named_profile["n_decisions"].sum() == 6830
assert named_profile["n_approved"].sum() == 6139
named_profile.sort_values("n_decisions", ascending=False).head(5)